# GCC Insurance Market Exploratory Data Analysis

### Business Task
Which GCC country should a hypothetical insurer prioritize for expansion, 
and having identified the UAE as a focus market what product-line 
trends should shape their UAE entry strategy?

### Notebook Structure
This notebook (`01_data_collection.ipynb`) covers:
1. Setup & Imports
2. Data Collection
   - 2.1 GCC Insurance Market Data (Alpen Capital)
   - 2.2 Macroeconomic Context Data (World Bank)
3. Data Quality Checks

Cleaning and merging continue in `02_cleaning.ipynb`; exploratory analysis 
and visualizations follow in `04_analysis.ipynb`.

### Data Sources
| Source | Coverage | Retrieved |
|---|---|---|
| Alpen Capital "GCC Insurance Industry" report, May 2026 | GWP, penetration, density, life/non-life split 6 GCC countries, 2025 (estimated) | Manual extraction, pages 39–45 |
| World Bank Open Data (World Development Indicators) | GDP, population, GDP per capita 6 GCC countries, 2010–2025 | API pull |

### Note on Source Selection

Several alternative sources were evaluated before settling on Alpen Capital's 
report: the World Bank's own insurance indicators (GFDD.DI.09/.10) were found 
to be discontinued for most countries after ~2015-2020, and Swiss Re's sigma 
explorer required an approval-gated registration with no guaranteed timeline. 
Alpen Capital's GCC-specific report (May 2026 edition) was selected as the 
primary insurance data source because it offered current (2025), 
region-specific, and consistently structured data across all six countries.

### Data Retrieval Date

In [34]:
from datetime import date
print(f"Notebook run/data verified on: {date.today()}")

Notebook run/data verified on: 2026-09-17


### 1. Setup & Imports

In [35]:
import pandas as pd
import requests

### 2. Data Collection 

### 2.1 GCC Insurance Market Data (Alpen Capital)

Manually extracted from the Alpen Capital "GCC Insurance Industry" report 
(May 20, 2026), pages 39–45 (Country-wise Market Size Forecast). Covers 
gross written premium (GWP), penetration, density, and life/non-life split 
for all six GCC countries as of 2025 (estimated). GCC market share is 
calculated from GWP rather than taken directly from the source.

In [36]:
INSURANCE_PATH = "data/raw/gcc_insurance_2025.csv"

gcc_insurance = pd.read_csv(INSURANCE_PATH)


assert gcc_insurance.shape == (6, 7), f"Unexpected shape: {gcc_insurance.shape}"


gcc_insurance["gcc_share_pct"] = (gcc_insurance["gwp_usd_billion"] / gcc_insurance["gwp_usd_billion"].sum() * 100).round(1)

gcc_insurance

,country,year,gwp_usd_billion,penetration_pct,density_usd,nonlife_share_pct,life_share_pct,gcc_share_pct
0,Saudi Arabia,2025,21.0,1.64,582.1,89.5,10.0,43.2
1,UAE,2025,20.5,3.58,1799.3,82.9,17.1,42.2
2,Qatar,2025,2.5,1.13,789.0,88.0,12.0,5.1
3,Kuwait,2025,2.3,1.45,448.8,91.3,8.7,4.7
4,Oman,2025,1.5,1.42,284.7,86.7,13.3,3.1
5,Bahrain,2025,0.8,1.65,484.0,87.5,10.0,1.6


### 2.2 Macroeconomic Context Data (World Bank)

Pulling GDP, population, and GDP per capita for the six GCC countries from 
the World Bank API (2010–2025). This provides the economic/demographic 
context needed to interpret the insurance penetration and density figures 
above e.g., whether market attractiveness correlates with wealth or 
population growth. Source: World Bank Open Data, World Development 
Indicators (indicators: NY.GDP.MKTP.CD, SP.POP.TOTL, NY.GDP.PCAP.CD).

In [37]:
countries = "are;sau;qat;kwt;bhr;omn"  


indicators = {
    "NY.GDP.MKTP.CD": "gdp_usd",
    "SP.POP.TOTL": "population",
    "NY.GDP.PCAP.CD": "gdp_per_capita"
}

all_data = []

for code, name in indicators.items():
    url = f"https://api.worldbank.org/v2/country/{countries}/indicator/{code}"
    params = {"format": "json", "per_page": 500, "date": "2010:2025"}
    response = requests.get(url, params=params)
    assert response.status_code == 200, f"API request failed: {response.status_code} for {code}"
    data = response.json()[1]  
    df = pd.json_normalize(data)
    df = df[["country.value", "date", "value"]]
    df.columns = ["country", "year", name]
    all_data.append(df)


worldbank_macro = all_data[0]
for df in all_data[1:]:
    worldbank_macro = worldbank_macro.merge(df, on=["country", "year"], how="outer")

worldbank_macro.head(20)

,country,year,gdp_usd,population,gdp_per_capita
0,Bahrain,2010,2.680598e+10,1228543,21819.329110
1,Bahrain,2011,2.991468e+10,1195020,25032.786774
2,Bahrain,2012,3.196340e+10,1208964,26438.673323
3,Bahrain,2013,3.382332e+10,1253191,26989.760115
4,Bahrain,2014,3.477253e+10,1314562,26451.796565
5,Bahrain,2015,3.252330e+10,1370322,23734.055114
6,Bahrain,2016,3.388468e+10,1423726,23800.001441
7,Bahrain,2017,3.720481e+10,1501116,24784.769351
8,Bahrain,2018,3.956798e+10,1503091,26324.406655
9,Bahrain,2019,4.044681e+10,1483756,27259.743860


### 3. Data Quality Checks

Before any cleaning or merging, we run systematic checks on both raw 
datasets to catch structural problems (missing values, duplicates, 
type mismatches) and logical inconsistencies (out-of-range percentages, 
implausible values) early, before they propagate into the analysis.

### 3.1 GCC Insurance Data Quality Checks

In [38]:
print("Shape:", gcc_insurance.shape)
print("\nData types:\n", gcc_insurance.dtypes)
print("\nMissing values per column:\n", gcc_insurance.isnull().sum())
print("\nDuplicate rows:", gcc_insurance.duplicated().sum())
print("\nDuplicate countries:", gcc_insurance["country"].duplicated().sum())

Shape: (6, 8)

Data types:
 country                  str
year                   int64
gwp_usd_billion      float64
penetration_pct      float64
density_usd          float64
nonlife_share_pct    float64
life_share_pct       float64
gcc_share_pct        float64
dtype: object

Missing values per column:
 country              0
year                 0
gwp_usd_billion      0
penetration_pct      0
density_usd          0
nonlife_share_pct    0
life_share_pct       0
gcc_share_pct        0
dtype: int64

Duplicate rows: 0

Duplicate countries: 0


In [39]:
gcc_insurance.describe()

,year,gwp_usd_billion,penetration_pct,density_usd,nonlife_share_pct,life_share_pct,gcc_share_pct
count,6.0,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000
mean,2025.0,8.100000,1.811667,731.316667,87.650000,11.850000,16.650000
std,0.0,9.818554,0.886734,548.931513,2.840951,3.048114,20.218877
min,2025.0,0.800000,1.130000,284.700000,82.900000,8.700000,1.600000
25%,2025.0,1.700000,1.427500,457.600000,86.900000,10.000000,3.500000
50%,2025.0,2.400000,1.545000,533.050000,87.750000,11.000000,4.900000
75%,2025.0,16.000000,1.647500,737.275000,89.125000,12.975000,32.925000
max,2025.0,21.000000,3.580000,1799.300000,91.300000,17.100000,43.200000


In [40]:
gcc_insurance["life_nonlife_sum"] = gcc_insurance["nonlife_share_pct"] + gcc_insurance["life_share_pct"]
gcc_insurance[["country", "nonlife_share_pct", "life_share_pct", "life_nonlife_sum"]]

,country,nonlife_share_pct,life_share_pct,life_nonlife_sum
0,Saudi Arabia,89.5,10.0,99.5
1,UAE,82.9,17.1,100.0
2,Qatar,88.0,12.0,100.0
3,Kuwait,91.3,8.7,100.0
4,Oman,86.7,13.3,100.0
5,Bahrain,87.5,10.0,97.5


### Finding: Saudi Arabia and Bahrain life + non-life shares do not sum to 100%

Saudi Arabia (99.5%) and Bahrain (97.5%) both fall slightly short of 100%, 
while the other four GCC countries sum exactly. Both figures were correctly 
transcribed from the source report's forecast charts (Alpen Capital, GCC 
Insurance Industry Report, May 2026, Exhibits 45 and 53), where individual 
life/non-life values are rounded to the nearest US$0.1 billion a small, 
recurring rounding artifact rather than a data entry error.

**Decision:** Both figures are retained as reported rather than forcing an 
artificial reconciliation, consistent with treating the source's own 
rounding as authoritative.

### Data Dictionary `gcc_insurance_2025.csv`

| Column | Type | Description | Unit |
|---|---|---|---|
| country | text | GCC country name | |
| year | integer | Reference year | |
| gwp_usd_billion | float | Gross Written Premium | US$ billions |
| penetration_pct | float | GWP as % of GDP | % |
| density_usd | float | GWP per capita | US$ |
| nonlife_share_pct | float | Non-life share of total GWP | % |
| life_share_pct | float | Life share of total GWP | % |
| gcc_share_pct | float | Share of total GCC GWP calculated as this country's GWP ÷ sum of all six countries' GWP, not taken directly from the source | % |

Note: unlike the World Bank dataset below, this CSV was created directly 
(values manually entered from the source report) rather than generated by 
this notebook so there is no corresponding save step here.

### 3.2 World Bank Macro Data Quality Checks

In [41]:
print("Shape:", worldbank_macro.shape)
print("\nData types:\n", worldbank_macro.dtypes)
print("\nMissing values per column:\n", worldbank_macro.isnull().sum())
print("\nDuplicate rows:", worldbank_macro.duplicated().sum())
print("\nYear range:", worldbank_macro["year"].min(), "-", worldbank_macro["year"].max())
print("\nCountries present:", sorted(worldbank_macro["country"].unique()))
print("\nRows per country:\n", worldbank_macro["country"].value_counts())

Shape: (96, 5)

Data types:
 country               str
year                  str
gdp_usd           float64
population          int64
gdp_per_capita    float64
dtype: object

Missing values per column:
 country           0
year              0
gdp_usd           1
population        0
gdp_per_capita    1
dtype: int64

Duplicate rows: 0

Year range: 2010 - 2025

Countries present: ['Bahrain', 'Kuwait', 'Oman', 'Qatar', 'Saudi Arabia', 'United Arab Emirates']

Rows per country:
 country
Bahrain                 16
Kuwait                  16
Oman                    16
Qatar                   16
Saudi Arabia            16
United Arab Emirates    16
Name: count, dtype: int64


In [42]:
worldbank_macro.describe()

,gdp_usd,population,gdp_per_capita
count,9.500000e+01,9.600000e+01,95.000000
mean,2.921243e+11,8.578134e+06,39764.102317
std,3.100968e+11,1.014624e+07,21137.148720
min,2.680598e+10,1.195020e+06,16784.863063
25%,8.773485e+10,2.535592e+06,24713.533614
50%,1.617400e+11,4.375871e+06,31707.873744
75%,4.064988e+11,9.254094e+06,49750.869218
max,1.276943e+12,3.697356e+07,108470.378825


In [43]:
worldbank_macro[worldbank_macro["gdp_usd"].isnull()]

,country,year,gdp_usd,population,gdp_per_capita
95,United Arab Emirates,2025,NaN,11513149,NaN


### Finding: UAE's 2025 GDP and GDP per capita are missing

`gdp_usd` and `gdp_per_capita` are both null for UAE, 2025 population 
for that same row is present, indicating World Bank has published 2025 
population estimates but not yet GDP figures for the UAE at time of 
retrieval. All other country-year combinations are complete.

**Decision:** to be finalized in `02_cleaning.ipynb` a consistent 
proxy-year approach will be chosen for all six countries' GDP figures 
(see cleaning notebook for the resolution).

### Findings: World Bank Data

1. **`year` column is stored as string, not integer.** This will need 
   conversion before any time-series operations (sorting, plotting, 
   filtering by range).

2. **Country naming inconsistency across datasets.** World Bank uses 
   "United Arab Emirates," while the Alpen Capital insurance dataset uses 
   "UAE." These will need to be standardized to a common naming convention 
   before the two datasets can be merged.

**Decision:** both issues will be addressed in the dedicated cleaning 
notebook (`02_cleaning.ipynb`), rather than patched here keeping data 
collection and data cleaning as separate steps.

### Data Dictionary `gcc_worldbank_macro.csv`

| Column | Type | Description | Unit |
|---|---|---|---|
| country | text | GCC country name | |
| year | text* | Reference year | |
| gdp_usd | float | Gross Domestic Product, current prices | US$ |
| population | integer | Total population | people |
| gdp_per_capita | float | GDP per capita | US$ |

*Note: `year` is stored as text in the raw file; converted to integer during 
cleaning (`02_cleaning.ipynb`).

In [44]:
print("Rows:", worldbank_macro.shape[0], "| Max year:", worldbank_macro["year"].max())
worldbank_macro.to_csv("data/raw/gcc_worldbank_macro.csv", index=False)

Rows: 96 | Max year: 2025


### Save Verified Data

Now that the World Bank data has been quality-checked and documented, save 
it to disk as the final raw-data output for this dataset the version 
`02_cleaning.ipynb` will load from.

### Summary Data Collection Complete

Two raw datasets collected and quality-checked:
- `data/raw/gcc_insurance_2025.csv` 6 rows, 8 columns, 1 known limitation 
  (Saudi Arabia and Bahrain life/non-life shares sum to slightly under 
  100%, documented above)
- `data/raw/gcc_worldbank_macro.csv` 96 rows, 5 columns, 3 known issues 
  to resolve in cleaning (year dtype, UAE naming mismatch, UAE 2025 GDP 
  not yet published)

**Next notebook:** `02_cleaning.ipynb` standardize country names, fix 
data types, decide on a consistent GDP reference year across all six 
countries, and merge both datasets into a single analysis-ready table.

### Known Limitations

- **Saudi Arabia and Bahrain life/non-life splits** sum to slightly under 
 100% a recurring rounding artifact in the source's chart values, 
  retained as reported (see Section 3.1 finding)
- **Insurance data is a single-year snapshot (2025 estimated)**, not a 
 full year-by-year time series trend analysis for the UAE deep-dive 
  will rely on a separate historical pull rather than this snapshot table
- **World Bank macro data has a gap for UAE's 2025 GDP**, which had not yet been 
published at the time of data collection. This is resolved in the cleaning 
step (`02_cleaning.ipynb`) by falling back to each country's most recently 
available year of macro data — UAE uses 2024 — with the actual year used 
tracked in a `macro_data_year` column for transparency.
- **Insurance figures were manually extracted from chart images** in a 
  PDF report (bar/line charts, not narrative text) rather than pulled 
 programmatically each value was individually read and cross-checked 
  against the chart labels, but this introduces a small manual-transcription 
  risk compared to API-sourced data